# TabPFN-3 fair test on a 16 GB GPU

**Before running (once):**
1. Settings panel (right) → **Accelerator: GPU T4 x2** (the script uses one T4, 15 GB). Avoid P100: it is slow for this model.
2. Settings → **Internet: On** (needed to install TabPFN and download its weights).
3. **Add Input** → your dataset containing `train_matrix.csv.gz`, `validation_matrix.csv.gz`, `test_matrix.csv.gz`, `sectors.csv` and `run_tabpfn3.py`.
4. **Add-ons → Secrets** → add `TABPFN_TOKEN` = your Prior Labs API key, and tick it for this notebook.
   (Create the key at platform.priorlabs.ai → Account; the TabPFN-3 license must be accepted there first.)

**To run in the background:** click **Save Version → Save & Run All (Commit)**. You can then close the browser; results appear in the version's *Output* tab.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q --upgrade tabpfn lightgbm

In [ ]:
import os, glob, shutil
from kaggle_secrets import UserSecretsClient

os.environ["TABPFN_TOKEN"] = UserSecretsClient().get_secret("TABPFN_TOKEN")
os.environ["TABPFN_NO_BROWSER"] = "1"

# Find the uploaded files wherever Kaggle mounted the dataset
train = glob.glob("/kaggle/input/**/train_matrix.csv.gz", recursive=True)
assert train, "train_matrix.csv.gz not found - did you add the dataset as input?"
DATA = os.path.dirname(train[0])
print("Data folder:", DATA, os.listdir(DATA))

script = glob.glob("/kaggle/input/**/run_tabpfn3.py", recursive=True)
assert script, "run_tabpfn3.py not found in the dataset"
shutil.copy(script[0], "/kaggle/working/run_tabpfn3.py")
SECTORS = os.path.join(DATA, "sectors.csv")
print("Sectors file:", SECTORS if os.path.exists(SECTORS) else "not found (sector features will be skipped)")

## Run 1: large context (fixed fit)
200,000 context rows (5x your laptop). If the GPU runs out of memory the script halves the rows automatically.
`--eval-every 5` predicts every 5th trading day: ~5x faster, same stock-picking test.

In [ ]:
!python /kaggle/working/run_tabpfn3.py --data-dir "$DATA" --sectors "$SECTORS" \
    --refit-on-train-val --context-rows 200000 --n-estimators 4 \
    --prediction-batch-size 5000 --eval-every 5 \
    --output /kaggle/working/results_200k.json --predictions /kaggle/working/predictions_200k.csv.gz

## Run 2 (optional, if time is left): large context sampled from 3 years
Same, but the 200k rows are spread over the last 3 years instead of the last ~6 months.

In [ ]:
!python /kaggle/working/run_tabpfn3.py --data-dir "$DATA" --sectors "$SECTORS" \
    --refit-on-train-val --context-rows 200000 --context-years 3 --n-estimators 4 \
    --prediction-batch-size 5000 --eval-every 5 \
    --output /kaggle/working/results_200k_3y.json --predictions /kaggle/working/predictions_200k_3y.csv.gz

In [ ]:
import json, os
for f in ["results_200k.json", "results_200k_3y.json"]:
    p = os.path.join("/kaggle/working", f)
    if not os.path.exists(p):
        continue
    r = json.load(open(p))
    print(f"\n{f}: context rows used {r['context_rows_used']:,}, features {r['n_features']}")
    for split in ("validation", "test"):
        for model, m in r[split].items():
            print(f"  {split:10s} {model:20s} AUC {m['roc_auc']:.2f}  within-day {m['within_day_auc']:.2f}  "
                  f"top-bottom {m['top_minus_bottom_pct']:+.3f}%  t {m['t_stat']:+.2f}")